In [5]:
# ══ CELLULE 1 — Téléchargement Bordeaux + Milan ══
import sys
sys.path.append('..')  # si le notebook est hors de wind_forecast_app/
from src.opendata import download_wind_dataset, diagnose_and_clean

# ── Bordeaux ── (vérifie que ces coordonnées correspondent à ton tout premier téléchargement)
LAT_BX, LON_BX = 44.8378, -0.5792
df_bordeaux_raw = download_wind_dataset(LAT_BX, LON_BX, "2015-01-01", "2025-12-31")
df_bordeaux, fixes_bx, warnings_bx = diagnose_and_clean(
    df_bordeaux_raw, datetime_col="time", target_col="ma_cible",
    secondary_col="variable_secondaire", temperature_col="temperature",
    pressure_col="pression", precipitation_col="precipitation",
    direction_col="direction")
print(f"Bordeaux : {len(df_bordeaux):,} lignes, vitesse moyenne {df_bordeaux['ma_cible'].mean():.2f} m/s")

# ── Milan ──
LAT_ML, LON_ML = 45.4642, 9.19
df_milan_raw = download_wind_dataset(LAT_ML, LON_ML, "2015-01-01", "2025-12-31")
df_milan, fixes_ml, warnings_ml = diagnose_and_clean(
    df_milan_raw, datetime_col="time", target_col="ma_cible",
    secondary_col="variable_secondaire", temperature_col="temperature",
    pressure_col="pression", precipitation_col="precipitation",
    direction_col="direction")
print(f"Milan : {len(df_milan):,} lignes, vitesse moyenne {df_milan['ma_cible'].mean():.2f} m/s")

# Sauvegarde — évite de retélécharger à chaque redémarrage du kernel
df_bordeaux.to_csv("bordeaux_wind_10y.csv", index=False)
df_milan.to_csv("milan_wind_10y.csv", index=False)

Bordeaux : 96,432 lignes, vitesse moyenne 5.12 m/s
Milan : 96,432 lignes, vitesse moyenne 3.28 m/s


In [7]:
# ══ CELLULE 2 — Imports du pipeline ══
from src.preprocessing import build_features
from src.splitting import causal_split
from src.training_calm import run_optuna_calm, train_full_pipeline_calm
from src.training import run_optuna, train_full_pipeline
from src.metrics import compute_metrics, metrics_by_bin
from src.model_selector import measure_wind_distribution
from sklearn.preprocessing import MinMaxScaler

In [8]:
# ══ CELLULE 3 — Modèle calme entraîné SUR Bordeaux ══
df_bx_ext, features_bx, _ = build_features(df_bordeaux, "time", "ma_cible",
    ["variable_secondaire","temperature","pression","precipitation","direction"])

zones_bx = causal_split(df_bx_ext)  # signature réelle : pas de features/target ici

scX_bx = MinMaxScaler().fit(zones_bx["gru_train"][features_bx])
scY_bx = MinMaxScaler().fit(zones_bx["gru_train"][["ma_cible"]])

best_params_calm = run_optuna_calm(zones_bx, features_bx, "ma_cible", scX_bx, scY_bx, n_trials=15)
results_calm_bx = train_full_pipeline_calm(zones_bx, features_bx, "ma_cible", best_params_calm, scX_bx, scY_bx)

mae_bx, rmse_bx, mape_bx, r2_bx = compute_metrics(results_calm_bx["true"], results_calm_bx["pred_combined"])
print(f"Modèle calme sur Bordeaux (entraînement natif) : MAE={mae_bx:.3f} RMSE={rmse_bx:.3f} MAPE={mape_bx:.2f}% R²={r2_bx:.4f}")

TypeError: tuple indices must be integers or slices, not str